# Topic: CTE (Common Table Expression) Pattern

## Definition (30-second explanation)
A Common Table Expression (CTE) is a named, temporary result set defined at the top of a SQL statement using the `WITH` keyword. It acts as a temporary virtual table that exists only for the duration of that single query, making complex logic significantly more readable by breaking it down step-by-step.

## Why Interviewers Ask This
Interviewers ask about CTEs to evaluate your ability to write readable, modular, and maintainable SQL. They want to see if you can break down complex business logic into logical steps rather than writing deeply nested, unreadable subqueries.

## Core Concepts
*   **Definition Sequence:** CTEs are defined sequentially at the top of the query. A CTE can reference any CTE defined *above* it, but cannot make forward references.
*   **Chaining:** Multiple CTEs are separated by commas, not semicolons. The `WITH` keyword is only used once at the beginning.
*   **Scope:** A CTE only exists for the single statement it is defined in and cannot be used in a subsequent separate query without redefining it.
*   **Recursion:** CTEs uniquely support recursive logic (e.g., hierarchical data) using `WITH RECURSIVE`.

## When to Use
*   Use CTEs for multi-step data transformations where intermediate logic is complex.
*   Use when you need to reference the same intermediate result set multiple times in the same query to avoid rewriting code.
*   Use for recursive queries, which cannot be done with standard subqueries.

## Advantages
*   **High Readability:** Logic is named and defined once at the top, reading naturally from top to bottom.
*   **Easy Debugging:** You can easily isolate and run a single CTE by changing the final `SELECT` statement to query just that CTE.
*   **Reusability:** A single CTE can be referenced multiple times within the main query.

## Limitations
*   **Materialization:** In most databases, CTEs are not automatically materialized (cached); the optimizer may just inline them, which can cause performance issues if an expensive CTE is referenced multiple times.
*   **Overhead on Simple Queries:** Using CTEs for simple, single-step filters is overkill and makes the code unnecessarily verbose.

## Common Comparisons
*   **CTE vs Subquery:** CTEs offer high readability, easy debugging, and reusability. Subqueries are nested (lower readability), hard to debug, must be repeated if reused, and do not support recursion.
*   **CTE vs Temp Table:** CTEs are query-scoped (disappear after the query runs). Temp tables persist for the entire session and can have indexes added to them, making them better for very large intermediate results.

## Common Interview Traps
*   **The Comma Trap:** Forgetting to use commas to separate multiple CTEs, or accidentally leaving a trailing comma before the final `SELECT`.
*   **Scope Errors:** Trying to use a CTE in a subsequent query statement without realizing it vanished after the first `SELECT`.
*   **Forward Referencing:** Trying to reference `cte_B` inside `cte_A` when `cte_B` is defined further down the chain.

## Python / SQL Syntax
```sql
-- Multiple CTEs (chained)
WITH 
cte_first AS (
    SELECT ... -- first CTE
    FROM ...
),
cte_second AS (
    SELECT ... -- second CTE can reference cte_first
    FROM cte_first
)
SELECT * 
FROM cte_second; -- final SELECT references any CTE
```

## 45-Second Interview Answer
A CTE, or Common Table Expression, is a named temporary result set defined with the `WITH` keyword at the start of a query. I use them primarily to replace complex, deeply nested subqueries. They make the code highly readable by breaking transformations into sequential, logical steps. Unlike subqueries, CTEs can be referenced multiple times in the main query and support recursion. However, for massive intermediate datasets that I need to query multiple times, I might opt for a Temp Table instead, as CTEs are usually query-scoped and not always materialized by the query optimizer.

## Example Questions:

### Q1: Using CTEs, find the top 3 customers by revenue in each region, and for each of those customers, show their most recent order.

**Ideal Interview Answer (MySQL):**
```sql
WITH CustomerRevenue AS (
    SELECT 
        region,
        customer_id,
        SUM(revenue) AS total_revenue
    FROM orders
    GROUP BY region, customer_id
),
RankedCustomers AS (
    SELECT 
        region,
        customer_id,
        total_revenue,
        DENSE_RANK() OVER(PARTITION BY region ORDER BY total_revenue DESC) as rev_rank
    FROM CustomerRevenue
),
TopCustomers AS (
    SELECT region, customer_id 
    FROM RankedCustomers 
    WHERE rev_rank <= 3
),
RecentOrders AS (
    SELECT 
        customer_id,
        order_date,
        order_details,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC) as recent_rank
    FROM orders
)
SELECT 
    t.region,
    t.customer_id,
    r.order_date,
    r.order_details
FROM TopCustomers t
JOIN RecentOrders r ON t.customer_id = r.customer_id
WHERE r.recent_rank = 1;
```

**Common Mistakes Candidates Make:**
*   Trying to do the aggregation (`SUM`), the ranking (`DENSE_RANK`), and the recent order logic (`ROW_NUMBER`) all in one or two messy subqueries instead of breaking them out.
*   Forgetting that Window Functions require an intermediate step to filter on the rank.

**Likely Interviewer Follow-up:**
"If the `orders` table has 10 billion rows, how might chaining these CTEs impact performance, and what alternative would you suggest?" (Answer: CTEs might be re-evaluated. Using a Temp Table with indexes for the aggregated revenue would be better).

### Q2: Write a 4-step CTE chain: (1) filter active users, (2) calculate each user's order count, (3) rank users by order count, (4) return top 20%.

**Ideal Interview Answer (MySQL):**
```sql
WITH ActiveUsers AS (
    SELECT user_id 
    FROM users 
    WHERE status = 'active'
),
UserOrderCount AS (
    SELECT 
        a.user_id,
        COUNT(o.order_id) AS order_count
    FROM ActiveUsers a
    LEFT JOIN orders o ON a.user_id = o.user_id
    GROUP BY a.user_id
),
RankedUsers AS (
    SELECT 
        user_id,
        order_count,
        PERCENT_RANK() OVER(ORDER BY order_count DESC) AS pct_rank
    FROM UserOrderCount
)
SELECT user_id, order_count
FROM RankedUsers
WHERE pct_rank <= 0.20;
```

**Common Mistakes Candidates Make:**
*   Using semicolons between the CTE steps instead of commas.
*   Calculating percentages manually (which is prone to integer division errors or edge cases) instead of utilizing the `PERCENT_RANK()` or `NTILE()` window functions.

**Likely Interviewer Follow-up:**
"Why did you use a `LEFT JOIN` in the second CTE instead of an `INNER JOIN`?" (Answer: To ensure active users with 0 orders are still included in the total population for accurate top 20% ranking).

### Q3: Rewrite this subquery using CTEs: SELECT * FROM (SELECT dept, AVG(sal) avg FROM emp GROUP BY dept) t JOIN departments d ON t.dept = d.dept_name WHERE t.avg > 80000.

**Ideal Interview Answer (MySQL):**
```sql
WITH DepartmentAverages AS (
    SELECT 
        dept, 
        AVG(sal) AS avg_sal
    FROM emp 
    GROUP BY dept
)
SELECT d.*, t.avg_sal
FROM DepartmentAverages t
JOIN departments d ON t.dept = d.dept_name
WHERE t.avg_sal > 80000;
```

**Common Mistakes Candidates Make:**
*   Adding `WITH` multiple times.
*   Including the `WHERE t.avg_sal > 80000` inside the CTE instead of keeping the CTE strictly to the aggregation, which reduces the CTE's reusability. 

**Likely Interviewer Follow-up:**
"In this specific, simple query, which is better: the derived table subquery or the CTE?" (Answer: It's subjective, but for a single-step simple query, the derived table might be perfectly fine. CTEs shine as complexity grows).

### Q4: Using CTEs, calculate a 7-day rolling average of daily sales. The final output should show: date, daily_sales, rolling_7day_avg.

**Ideal Interview Answer (MySQL):**
```sql
WITH DailySales AS (
    SELECT 
        order_date,
        SUM(sales_amount) AS daily_sales
    FROM sales
    GROUP BY order_date
)
SELECT 
    order_date,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY order_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg
FROM DailySales
ORDER BY order_date;
```

**Common Mistakes Candidates Make:**
*   Forgetting to aggregate the data down to the daily level first (in the CTE). Applying the rolling average window function directly to transaction-level data will yield a per-transaction rolling average, not a daily one.
*   Using `ROWS BETWEEN 7 PRECEDING` instead of `6 PRECEDING` (6 prior days + current day = 7 days).

**Likely Interviewer Follow-up:**
"What happens to the rolling average on the first 6 days of the dataset?" (Answer: It will calculate the average of whatever days are available—e.g., a 3-day average on day 3. We might need a `CASE` statement to return NULL if a full 7 days aren't available).